> **SUPERSEDED - MERRA2_Square_Export_Colab.ipynb now exports all 15 AER bands for
> 2010-2024 with the UTC-6 window, so this separate surface export is not needed.**

# MERRA-2 AER surface + extinction gap fill, 2010-2024

Exports the ten AER bands that are missing outside Ontario:

```
BCSMASS, OCSMASS, SO4SMASS, SSSMASS25, DUSMASS25      (surface mass)
BCEXTTAU, OCEXTTAU, SUEXTTAU, DUEXTTAU, TOTEXTTAU     (extinction)
```

The column-mass bands are **not** re-exported - they are already complete in
`SourceRegion_MERRA_2010_2024`. FLX and SLV are not touched either.

Covers the whole square for every day 2010-01-01 .. 2024-12-31, so the south
strip (38.5-41 N) and 2010-2011 outside Ontario are filled in one consistent
source rather than spliced.

## Grid

Pinned to the completed archive's lattice, not to a scale in metres:

```
crsTransform = [0.4940734062657368, 0, -100.29690147194458,
                0, -0.4940734062657368, 60.77102897068563]
```

71 x 48 whole pixels, EPSG:4326, bounds 100.297 W - 65.218 W, 37.056 N - 60.771 N.
The region is computed from those constants, so it lands on pixel boundaries and
no edge pixel is half-masked.

Daily value = mean of that UTC day's hourly images. Empty days are written as
**-999**. Bands are named `YYYYMMDD_<VAR>`; one GeoTIFF per month.


In [ ]:
!pip install -q earthengine-api geemap

In [ ]:
import calendar
import time
import ee

PROJECT_ID = 'my-project-pm25predictionon'  # change this

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

print('Earth Engine ready')

In [ ]:
START_YEAR = 2010
END_YEAR = 2024
DRIVE_FOLDER = 'SourceRegion_MERRA_AER_SFC_2010_2024'
ASSET = 'NASA/GSFC/MERRA/aer/2'

# ---- grid of the completed archive (SourceRegion_MERRA_2010_2024) --------
PX = 0.4940734062657368
ULX = -100.29690147194458        # upper-left corner
ULY = 60.77102897068563
WIDTH, HEIGHT = 71, 48

REGION_BBOX = [ULX, ULY - HEIGHT * PX, ULX + WIDTH * PX, ULY]
ROI = ee.Geometry.Rectangle(REGION_BBOX, geodesic=False)
CRS = 'EPSG:4326'
CRS_TRANSFORM = [PX, 0, ULX, 0, -PX, ULY]

MAX_PIXELS = 1_000_000_000_000
SUBMIT_SLEEP = 0.3

# surface mass + extinction only; the *CMASS column bands are already done
VARIABLES = [
    'BCSMASS', 'OCSMASS', 'SO4SMASS', 'SSSMASS25', 'DUSMASS25',
    'BCEXTTAU', 'OCEXTTAU', 'SUEXTTAU', 'DUEXTTAU', 'TOTEXTTAU',
]

print('region :', [round(v, 6) for v in REGION_BBOX])
print('grid   :', WIDTH, 'x', HEIGHT, 'pixels')
print('bands  :', len(VARIABLES), 'per day')

# ---- fail early if any band name is wrong --------------------------------
_bands = ee.ImageCollection(ASSET).filterBounds(ROI).first().bandNames().getInfo()
_missing = [v for v in VARIABLES if v not in _bands]
assert not _missing, f'bands not found in {ASSET}: {_missing}'
print(f'{len(VARIABLES)} bands verified on {ASSET}')

# show what we are deliberately not exporting (should include the 5 CMASS)
_extra = [b for b in _bands if b not in VARIABLES]
print('available but not exported:', _extra)

In [ ]:
def aer_daily_image(date: ee.Date) -> ee.Image:
    start = ee.Date(date)
    end = start.advance(1, 'day')
    coll = (
        ee.ImageCollection(ASSET)
        .filterDate(start, end)
        .filterBounds(ROI)
        .select(VARIABLES)
    )
    reduced = coll.mean()
    return ee.Image(ee.Algorithms.If(coll.size().gt(0), reduced,
                                     empty_daily_image(start))).set(
        'system:time_start', start.millis()
    )


def empty_daily_image(date: ee.Date) -> ee.Image:
    # -999, never 0: an absent day must not look like a measured zero
    return ee.Image.constant([-999] * len(VARIABLES)).toFloat().rename(VARIABLES).set(
        'system:time_start', date.millis()
    )


def aer_month_stack(year: int, month: int) -> ee.Image:
    start = ee.Date.fromYMD(year, month, 1)
    n_days = calendar.monthrange(year, month)[1]
    stack = None
    for i in range(n_days):
        d = start.advance(i, 'day')
        tag = d.format('YYYYMMdd')
        daily = aer_daily_image(d)
        renamed = daily.rename([tag.cat('_' + v) for v in VARIABLES])
        stack = renamed if stack is None else stack.addBands(renamed)
    return stack.toFloat()


def export_month(year: int, month: int) -> None:
    tag = f'{year:04d}_{month:02d}'
    task = ee.batch.Export.image.toDrive(
        image=aer_month_stack(year, month),
        description=f'SRC_MERRA_AERSFC_{tag}',
        folder=DRIVE_FOLDER,
        fileNamePrefix=f'source_region_merra_aersfc_daily_stack_{tag}',
        region=ROI,
        crs=CRS,
        crsTransform=CRS_TRANSFORM,
        maxPixels=MAX_PIXELS,
        fileFormat='GeoTIFF',
    )
    task.start()
    print('started', tag)

In [ ]:
# ---- smoke test ---------------------------------------------------------
# 2010-01: 31 days x 10 bands = 310 bands on a 71 x 48 grid.
# Check the task, then inspect the file before running the submit cell.

export_month(2010, 1)
print('expected: 310 bands, first 20100101_BCSMASS, last 20100131_TOTEXTTAU')

In [ ]:
# ---- submit all 180 months ---------------------------------------------
# 15 years x 12 months = 180 tasks, roughly a minute to submit.
# Only run after the smoke test looks right.

for year in range(START_YEAR, END_YEAR + 1):
    for month in range(1, 13):
        export_month(year, month)
        time.sleep(SUBMIT_SLEEP)
print('all 180 submitted')